# Cortex Analyst: Access Control & Row-Level Security

**Purpose**: Examples and templates for RBAC setup for Cortex Analyst and row access policy propagation to semantic views.

**Target Audience**: Snowflake Administrators

**Last Updated**: March 2026

**Version**: 2.0

---

## Overview

This runbook demonstrates minimal Cortex Analyst setup with role separation, row access policies that propagate through semantic views, and testing via the Snowflake UI.

---

## Configuration Variables

In [ ]:
USE SECONDARY ROLES NONE;

-- ============================================================================
-- CONFIGURATION VARIABLES
-- ============================================================================

-- Database and Schema
SET DATABASE_NAME = 'RUNBOOKS_DB';
SET SCHEMA_NAME = 'CORTEX_ANALYST';

-- Full qualified paths (required for IDENTIFIER() function)
SET FULL_SCHEMA_PATH = 'RUNBOOKS_DB.CORTEX_ANALYST';

-- Role Configuration
SET ANALYST_ROLE = 'CORTEX_ANALYST_ROLE';  -- Role for querying semantic views
SET SEMANTIC_VIEW_ADMIN_ROLE = 'SEMANTIC_VIEW_ADMIN';  -- Role for creating semantic views

-- Warehouse
SET WAREHOUSE_NAME = 'COMPUTE_WH';

-- Display configuration
SELECT 
    $DATABASE_NAME AS DATABASE_NAME,
    $SCHEMA_NAME AS SCHEMA_NAME,
    $FULL_SCHEMA_PATH AS FULL_SCHEMA_PATH,
    $ANALYST_ROLE AS ANALYST_ROLE,
    $SEMANTIC_VIEW_ADMIN_ROLE AS SEMANTIC_VIEW_ADMIN_ROLE,
    $WAREHOUSE_NAME AS WAREHOUSE_NAME;


---

## Part 1: Model-Level RBAC for Cortex Analyst

Model-level RBAC controls access to specific LLMs used by Cortex Analyst. Cortex Analyst automatically assigns each request to a model -- you cannot choose which model is used. To restrict model access, use `CORTEX_MODELS_ALLOWLIST` (see Notebook 01) or model application roles.

The `SNOWFLAKE.CORTEX_ANALYST_USER` database role grants access to Cortex Analyst. By default, `CORTEX_USER` (granted to `PUBLIC`) includes analyst access.

## Part 2: Minimal Cortex Analyst Setup with Role Separation

Two roles:
- **Semantic View Admin** (`SEMANTIC_VIEW_ADMIN`): Creates semantic views, grants access
- **Cortex Analyst Role** (`CORTEX_ANALYST_ROLE`): Queries semantic views via Cortex Analyst

### Step 1: Create Environment

In [ ]:
-- Create database, schema, and warehouse
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS IDENTIFIER($DATABASE_NAME);
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($FULL_SCHEMA_PATH);

CREATE WAREHOUSE IF NOT EXISTS IDENTIFIER($WAREHOUSE_NAME)
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

### Step 2: Create and Configure Cortex Analyst Role

In [ ]:
-- Create custom role for Cortex Analyst users
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS IDENTIFIER($ANALYST_ROLE)
    COMMENT = 'Role for Cortex Analyst users with minimal privileges';

-- Grant CORTEX_ANALYST_USER database role
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE IDENTIFIER($ANALYST_ROLE);

-- Assign both to Sysadmin (for demo purposes)

GRANT ROLE IDENTIFIER($ANALYST_ROLE) TO ROLE SYSADMIN;

### Step 2b: Create Semantic View Admin Role

In [ ]:
-- Create role for semantic view administration
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE)
    COMMENT = 'Role for creating and managing semantic views';

-- Grant necessary privileges for creating semantic views
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);

-- Grant CREATE SEMANTIC VIEW privilege
GRANT CREATE SEMANTIC VIEW ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);

-- Grant SELECT on tables (needed to create semantic views on them)
GRANT SELECT ON ALL TABLES IN SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);
GRANT SELECT ON FUTURE TABLES IN SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);

GRANT ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE) to ROLE SYSADMIN;


### Step 3: Grant Minimal Object Privileges

In [ ]:
-- Grant minimal required privileges
USE ROLE ACCOUNTADMIN;

-- Database and schema access
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($ANALYST_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($ANALYST_ROLE);

-- Warehouse access
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($ANALYST_ROLE);

---

## Part 3: Row Access Policies with Semantic Views

Row access policies on base tables automatically propagate to semantic views built on those tables. Semantic views use owner's rights, so users don't need direct table access.

### Step 4: Create Two Identical Tables

1. **sales_data_unrestricted** - No row access policy (all users see all data)
2. **sales_data_restricted** - With row access policy (filtered by role)

In [ ]:
-- Create first table WITHOUT row access policy
USE ROLE ACCOUNTADMIN;
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

CREATE OR REPLACE TABLE sales_data_unrestricted (
    sale_id INT,
    product_name STRING,
    region STRING,
    amount DECIMAL(10,2),
    sale_date DATE,
    sales_rep_role STRING
);

-- Insert sample data
INSERT INTO sales_data_unrestricted VALUES
    (1, 'Laptop', 'West', 1200.00, '2024-01-15', 'CORTEX_ANALYST_ROLE'),
    (2, 'Mouse', 'West', 25.00, '2024-01-16', 'CORTEX_ANALYST_ROLE'),
    (3, 'Keyboard', 'East', 75.00, '2024-01-17', 'SALES_MANAGER'),
    (4, 'Monitor', 'East', 350.00, '2024-01-18', 'SALES_MANAGER'),
    (5, 'Desk', 'West', 450.00, '2024-01-19', 'CORTEX_ANALYST_ROLE'),
    (6, 'Chair', 'Central', 200.00, '2024-01-20', 'SALES_MANAGER'),
    (7, 'Tablet', 'West', 600.00, '2024-01-21', 'CORTEX_ANALYST_ROLE'),
    (8, 'Phone', 'East', 800.00, '2024-01-22', 'SALES_MANAGER');

SELECT 'Unrestricted table created with ' || COUNT(*) || ' rows' AS STATUS
FROM sales_data_unrestricted;


In [ ]:
-- Create second table (will have row access policy applied later)
CREATE OR REPLACE TABLE sales_data_restricted (
    sale_id INT,
    product_name STRING,
    region STRING,
    amount DECIMAL(10,2),
    sale_date DATE,
    sales_rep_role STRING
);

-- Insert identical sample data
INSERT INTO sales_data_restricted VALUES
    (1, 'Laptop', 'West', 1200.00, '2024-01-15', 'CORTEX_ANALYST_ROLE'),
    (2, 'Mouse', 'West', 25.00, '2024-01-16', 'CORTEX_ANALYST_ROLE'),
    (3, 'Keyboard', 'East', 75.00, '2024-01-17', 'SALES_MANAGER'),
    (4, 'Monitor', 'East', 350.00, '2024-01-18', 'SALES_MANAGER'),
    (5, 'Desk', 'West', 450.00, '2024-01-19', 'CORTEX_ANALYST_ROLE'),
    (6, 'Chair', 'Central', 200.00, '2024-01-20', 'SALES_MANAGER'),
    (7, 'Tablet', 'West', 600.00, '2024-01-21', 'CORTEX_ANALYST_ROLE'),
    (8, 'Phone', 'East', 800.00, '2024-01-22', 'SALES_MANAGER');

SELECT 'Restricted table created with ' || COUNT(*) || ' rows' AS STATUS
FROM sales_data_restricted;


### Step 5: Create and Apply Row Access Policy

In [ ]:
-- Create row access policy based on CURRENT_ROLE()
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE ROW ACCESS POLICY sales_row_policy 
    AS (sales_rep_role STRING) 
    RETURNS BOOLEAN ->
        CASE
            -- ACCOUNTADMIN can see all data
            WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN TRUE
            -- Users can only see rows where sales_rep_role matches their current role
            WHEN CURRENT_ROLE() = sales_rep_role THEN TRUE
            --
            WHEN CURRENT_ROLE() = 'MCP_USER_ROLE' AND sales_rep_role = 'SALES_MANAGER' THEN TRUE -- for testing MCP
            -- Default: no access
            ELSE FALSE
        END
    COMMENT = 'Row access policy: users see only data for their role';


In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
-- Apply the row access policy to the restricted table only

ALTER TABLE sales_data_restricted 
    ADD ROW ACCESS POLICY sales_row_policy ON (sales_rep_role);

### Step 6: Create Semantic Views

Two semantic views -- one per table. The restricted table's row access policy automatically applies.

In [ ]:
-- Create semantic view on UNRESTRICTED table
-- Using SEMANTIC_VIEW_ADMIN_ROLE to demonstrate role separation
USE ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

CREATE OR REPLACE SEMANTIC VIEW sales_semantic_view_unrestricted
    TABLES (
        sales AS sales_data_unrestricted
        PRIMARY KEY (sale_id)
    )
    FACTS (
        sales.sale_id AS sale_id,
        sales.amount AS amount
    )
    DIMENSIONS (
        sales.product_name AS product_name,
        sales.region AS region,
        sales.sale_date AS sale_date,
        sales.sales_rep_role AS sales_rep_role
    )
    COMMENT = 'Semantic view WITHOUT row access policy - all users see all data';



In [ ]:
-- Create semantic view on RESTRICTED table (with row access policy)
-- Still using SEMANTIC_VIEW_ADMIN_ROLE
USE ROLE IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);

CREATE OR REPLACE SEMANTIC VIEW sales_semantic_view_restricted
    TABLES (
        sales AS sales_data_restricted
        PRIMARY KEY (sale_id)
    )
    FACTS (
        sales.sale_id AS sale_id,
        sales.amount AS amount
    )
    DIMENSIONS (
        sales.product_name AS product_name,
        sales.region AS region,
        sales.sale_date AS sale_date,
        sales.sales_rep_role AS sales_rep_role
    )
    COMMENT = 'Semantic view WITH row access policy - users see only their authorized data';

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA IDENTIFIER($FULL_SCHEMA_PATH);

GRANT REFERENCES, SELECT ON SEMANTIC VIEW sales_semantic_view_restricted TO ROLE IDENTIFIER($ANALYST_ROLE);

GRANT SELECT ON TABLE RUNBOOKS_DB.CORTEX_ANALYST.SALES_DATA_RESTRICTED TO ROLE IDENTIFIER($ANALYST_ROLE); -- needed when querying through Cortex Analyst Interface

---

## Part 4: Testing Your Semantic Views

### Access Through Snowflake UI

1. Navigate to **Data > Databases > RUNBOOKS_DB > CORTEX_ANALYST**
2. Find `SALES_SEMANTIC_VIEW_UNRESTRICTED` and `SALES_SEMANTIC_VIEW_RESTRICTED`

### Testing Row Access Policies

1. Open a semantic view in the Snowflake UI
2. Ask: "Show me all sales data"
3. Switch roles between `CORTEX_ANALYST_ROLE` and `ACCOUNTADMIN`
4. Compare results: `CORTEX_ANALYST_ROLE` sees 4 rows (region = 'West'), `ACCOUNTADMIN` sees all 8 rows

## Part 5: Cleanup (Optional)

Run these commands to clean up all resources created in this runbook.

In [ ]:
-- Cleanup: Drop all created objects
--USE ROLE ACCOUNTADMIN;

-- Drop semantic views
--DROP SEMANTIC VIEW IF EXISTS sales_semantic_view_unrestricted;
--DROP SEMANTIC VIEW IF EXISTS sales_semantic_view_restricted;

-- Drop tables (this also removes the row access policy)
--DROP TABLE IF EXISTS sales_data_unrestricted;
--DROP TABLE IF EXISTS sales_data_restricted;

---- Drop row access policy
--DROP ROW ACCESS POLICY IF EXISTS sales_row_policy;

-- Drop roles
--DROP ROLE IF EXISTS IDENTIFIER($ANALYST_ROLE);
--DROP ROLE IF EXISTS IDENTIFIER($SEMANTIC_VIEW_ADMIN_ROLE);

-- Drop warehouse
--DROP WAREHOUSE IF EXISTS IDENTIFIER($WAREHOUSE_NAME);

-- Drop schema and database (commented out for safety)
-- DROP SCHEMA IF EXISTS IDENTIFIER($FULL_SCHEMA_PATH);
-- DROP DATABASE IF EXISTS IDENTIFIER($DATABASE_NAME);

---

## Additional Resources

- [Cortex Analyst Overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)
- [Access Control Requirements](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst#access-control-requirements)
- [Row Access Policies](https://docs.snowflake.com/en/user-guide/security-row-intro)
- [Database Roles](https://docs.snowflake.com/en/user-guide/security-access-control-overview#database-roles)

---

**Version:** 2.0  
**Last Updated:** March 2026